# 02. 전처리 및 그룹 할당

**목적**
1. 5개 데이터 통합
2. 건강 수치를 정상(0)/주의(1)/위험(2)으로 변환
3. 변환 결과 조합으로 규칙 기반 그룹 할당

**그룹 정의**
- 정상군
- 고혈압군
- 혈당이상군
- 비만군
- 고혈압+혈당이상 복합군
- 고혈압+비만 복합군
- 혈당이상+비만 복합군
- 복합위험군 (3가지 이상)

## 0. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings('ignore')

## 1. 데이터 로드 및 통합

In [ ]:
# ── 1-1. 국민건강보험공단 ──────────────────────────────────────
nhis = pd.read_csv('../../data/raw/국민건강보험공단_건강검진정보_2024.CSV.csv', encoding='cp949')
nhis = nhis.rename(columns={
    '성별코드': 'gender',
    '연령대코드(5세단위)': 'age_group',
    '신장(5cm단위)': 'height',
    '체중(5kg단위)': 'weight',
    '허리둘레': 'waist',
    '수축기혈압': 'sbp',
    '이완기혈압': 'dbp',
    '식전혈당(공복혈당)': 'fbs',
})
nhis['bmi'] = (nhis['weight'] / ((nhis['height'] / 100) ** 2)).round(1)
nhis_std = pd.DataFrame({
    'sbp':    nhis['sbp'],
    'dbp':    nhis['dbp'],
    'fbs':    nhis['fbs'],
    'bmi':    nhis['bmi'],
    'waist':  nhis['waist'],
    'gender': nhis['gender'],
    'source': 'NHIS',
})
print(f'NHIS:              {len(nhis_std):>7,}명')

In [ ]:
# ── 1-2. Personalized Diet Recommendations ────────────────────
# 컬럼: Blood_Pressure_Systolic, Blood_Pressure_Diastolic,
#       Blood_Sugar_Level, BMI, Recommended_Meal_Plan
diet1 = pd.read_csv('../../data/raw/Personalized_Diet_Recommendations.csv')
diet1_std = pd.DataFrame({
    'sbp':    diet1['Blood_Pressure_Systolic'],
    'dbp':    diet1['Blood_Pressure_Diastolic'],
    'fbs':    diet1['Blood_Sugar_Level'],
    'bmi':    diet1['BMI'],
    'waist':  np.nan,
    'gender': np.nan,
    'source': 'DIET1',
})
print(f'Personalized Diet: {len(diet1_std):>7,}명')

In [ ]:
# ── 1-3. Diet Recommendations Dataset ────────────────────────
# 컬럼: Blood_Pressure_mmHg, Glucose_mg/dL, BMI,
#       Disease_Type, Severity, Diet_Recommendation
diet2 = pd.read_csv('../../data/raw/diet_recommendations_dataset.csv')
diet2_std = pd.DataFrame({
    'sbp':    diet2['Blood_Pressure_mmHg'],
    'dbp':    np.nan,
    'fbs':    diet2['Glucose_mg/dL'],
    'bmi':    diet2['BMI'],
    'waist':  np.nan,
    'gender': np.nan,
    'source': 'DIET2',
})
print(f'Diet Recomm:       {len(diet2_std):>7,}명')

In [ ]:
# ── 1-4. Cardiovascular Disease Dataset ──────────────────────
# 컬럼: ap_hi(sbp), ap_lo(dbp), height, weight, gender, cardio
cardio = pd.read_csv('../../data/raw/cardio_train.csv', sep=';')
cardio['bmi'] = (cardio['weight'] / ((cardio['height'] / 100) ** 2)).round(1)
cardio_std = pd.DataFrame({
    'sbp':    cardio['ap_hi'],
    'dbp':    cardio['ap_lo'],
    'fbs':    np.nan,
    'bmi':    cardio['bmi'],
    'waist':  np.nan,
    'gender': cardio['gender'],
    'source': 'CARDIO',
})
print(f'Cardiovascular:    {len(cardio_std):>7,}명')

In [ ]:
# ── 1-5. Pima Indians Diabetes ───────────────────────────────
# 컬럼: Glucose(fbs), BloodPressure(dbp), BMI, Outcome(당뇨진단)
diabetes = pd.read_csv('../../data/raw/diabetes.csv')
diabetes_std = pd.DataFrame({
    'sbp':    np.nan,
    'dbp':    diabetes['BloodPressure'],
    'fbs':    diabetes['Glucose'],
    'bmi':    diabetes['BMI'],
    'waist':  np.nan,
    'gender': np.nan,
    'source': 'DIABETES',
})
print(f'Pima Diabetes:     {len(diabetes_std):>7,}명')

In [ ]:
# ── 1-6. 통합 ─────────────────────────────────────────────────
df = pd.concat([nhis_std, diet1_std, diet2_std, cardio_std, diabetes_std], ignore_index=True)
print(f'통합 데이터: {len(df):,}명')
print(df['source'].value_counts())

## 2. 이상치 제거

In [ ]:
OUTLIER_BOUNDS = {
    'sbp':   (60, 300),
    'dbp':   (40, 200),
    'fbs':   (50, 600),
    'bmi':   (10, 60),
    'waist': (40, 200),
}
before = len(df)
for col, (lo, hi) in OUTLIER_BOUNDS.items():
    mask = df[col].isna() | ((df[col] >= lo) & (df[col] <= hi))
    df = df[mask]
print(f'이상치 제거: {before:,} → {len(df):,}명')

## 3. 정상/주의/위험 변환

국민건강보험공단 및 대한의학회 기준

| 항목 | 정상(0) | 주의(1) | 위험(2) |
|---|---|---|---|
| 수축기혈압 | 120 미만 | 120~139 | 140 이상 |
| 이완기혈압 | 80 미만 | 80~89 | 90 이상 |
| 공복혈당 | 100 미만 | 100~125 | 126 이상 |
| BMI | 23 미만 | 23~24.9 | 25 이상 |
| 허리둘레 | 남90/여85 미만 | - | 남90/여85 이상 |

In [ ]:
def classify_sbp(v):
    if pd.isna(v): return np.nan
    if v < 120:   return 0
    elif v < 140: return 1
    else:         return 2

def classify_dbp(v):
    if pd.isna(v): return np.nan
    if v < 80:   return 0
    elif v < 90: return 1
    else:        return 2

def classify_fbs(v):
    if pd.isna(v): return np.nan
    if v < 100:   return 0
    elif v < 126: return 1
    else:         return 2

def classify_bmi(v):
    if pd.isna(v): return np.nan
    if v < 23:   return 0
    elif v < 25: return 1
    else:        return 2

def classify_waist(row):
    if pd.isna(row['waist']): return np.nan
    if row['gender'] == 2:   return 2 if row['waist'] >= 85 else 0
    else:                    return 2 if row['waist'] >= 90 else 0

df['sbp_c']   = df['sbp'].apply(classify_sbp)
df['dbp_c']   = df['dbp'].apply(classify_dbp)
df['fbs_c']   = df['fbs'].apply(classify_fbs)
df['bmi_c']   = df['bmi'].apply(classify_bmi)
df['waist_c'] = df.apply(classify_waist, axis=1)

print('변환 완료')
for col in ['sbp_c', 'dbp_c', 'fbs_c', 'bmi_c', 'waist_c']:
    print(f'\n[{col}]')
    print(df[col].value_counts().sort_index().rename({0.0:'정상', 1.0:'주의', 2.0:'위험'}))

## 4. 규칙 기반 그룹 할당

혈압(sbp/dbp), 혈당(fbs), 비만(bmi/waist) 3가지 축으로 그룹 정의

In [ ]:
def assign_group(row):
    # 혈압 이상 여부: sbp 또는 dbp 주의/위험
    hypertension = (row['sbp_c'] >= 1) or (row['dbp_c'] >= 1)

    # 혈당 이상 여부: fbs 주의/위험
    glucose = (row['fbs_c'] >= 1)

    # 비만 여부: bmi 주의/위험 또는 waist 위험
    obesity = (row['bmi_c'] >= 1) or (row['waist_c'] >= 1)

    count = sum([hypertension, glucose, obesity])

    if count == 0:
        return '정상군'
    elif count == 3:
        return '복합위험군'
    elif hypertension and glucose:
        return '고혈압+혈당이상군'
    elif hypertension and obesity:
        return '고혈압+비만군'
    elif glucose and obesity:
        return '혈당이상+비만군'
    elif hypertension:
        return '고혈압군'
    elif glucose:
        return '혈당이상군'
    else:
        return '비만군'

df['group'] = df.apply(assign_group, axis=1)

print('=== 그룹별 인원 ===')
print(df['group'].value_counts())

In [ ]:
# 그룹별 인원 시각화
group_counts = df['group'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ECC71', '#E74C3C', '#E67E22', '#3498DB', '#9B59B6', '#1ABC9C', '#F39C12', '#E74C3C']
bars = ax.barh(group_counts.index, group_counts.values,
               color=colors[:len(group_counts)], edgecolor='white')
for bar, val in zip(bars, group_counts.values):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
            f'{val:,}명 ({val/len(df)*100:.1f}%)', va='center', fontsize=9)
ax.set_title('그룹별 인원 분포', fontsize=13, fontweight='bold')
ax.set_xlabel('인원 수')
plt.tight_layout()
plt.show()

In [ ]:
# 그룹별 원본 수치 평균
print('=== 그룹별 평균 건강 수치 ===')
print(df.groupby('group')[['sbp', 'dbp', 'fbs', 'bmi', 'waist']].mean().round(1))

In [ ]:
# 그룹별 위험 비율 히트맵
risk_ratio = df.groupby('group')[['sbp_c', 'dbp_c', 'fbs_c', 'bmi_c', 'waist_c']].apply(
    lambda g: (g == 2).mean() * 100
).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(risk_ratio, annot=True, fmt='.1f', cmap='YlOrRd',
            vmin=0, vmax=100, ax=ax,
            xticklabels=['수축기혈압', '이완기혈압', '공복혈당', 'BMI', '허리둘레'])
ax.set_title('그룹별 위험 비율(%) 히트맵', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 저장

In [ ]:
os.makedirs('../../data/processed', exist_ok=True)

df.to_csv('../../data/processed/health_grouped.csv', index=False, encoding='utf-8-sig')

print('저장 완료')
print('  data/processed/health_grouped.csv  → 03_classification 입력')

## 6. 결과 요약

In [ ]:
print('=' * 60)
print('전처리 및 그룹 할당 결과 요약')
print('=' * 60)
print(f'통합 데이터 규모: {len(df):,}명')
print()
print('=== 그룹별 인원 ===')
for group, count in df['group'].value_counts().items():
    print(f'  {group:20s}: {count:>7,}명 ({count/len(df)*100:.1f}%)')
print()
print('→ 다음 단계: 03_classification.ipynb')
print('  group 컬럼을 타겟으로 식단 플랜 분류 모델 학습')
print('=' * 60)